# CUDA Source Notebook

In [1]:
%%bash
nvidia-smi

Thu Oct 15 20:54:05 2026       
+-------------------------------------------------------------------

----------------------+
| NVIDIA-SMI 610.57.01              KMD Version: 610.88        CUDA UMD Vers

ion: 13.3     |
+-----------------------------------------+------------------------+----------------

------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       

...

ess name                        GPU Memory |
|        ID   ID                                       

                        Usage      |
|==============================================================

===========================|


|    0   N/A  N/A              25      G   /Xwayland                             N/A      |
+-------

----------------------------------------------------------------------------------+


In [2]:
%%bash
for file in *.cpp *.hpp *.cu *.cuh *.so *.py *.sh *.txt threads __pycache__; do
    rm -rf "$file"
done

## 101: The CUDA Kernel

In [3]:
%%writefile parse_num.hpp
#pragma once
#include <cerrno>
#include <charconv>
#include <string_view>
#include <system_error>

template <typename T>
int parse_num(T& val, const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  auto [read_end, read_err] = std::from_chars(arg.data(), arg_end, val);
  bool parse_err = read_err != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_err) {
    return static_cast<int>(read_err);
  }
  if (partial_read) {
    return EINVAL;
  }
  return 0;
}


Writing parse_num.hpp


In [4]:
%%writefile get_thread.cuh
#pragma once
#include <cstdint>

__device__
inline uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing get_thread.cuh


In [5]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdint>
#include <cstdio>
#include "get_thread.cuh"

__global__
static void threads_kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [6]:
%%writefile threads.cuh
#pragma once
#include <cstdint>
#include <cuda_runtime.h>
#include "threads_kernel.cuh"

static inline cudaError_t threads(uint32_t grid_size, uint32_t block_size) {
  constexpr uint32_t device_id = 0;

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    return set_device_err;
  }

  threads_kernel<<<grid_size, block_size>>>();
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }

  return cudaDeviceSynchronize();
}

Writing threads.cuh


In [7]:
%%writefile threads.cu
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <print>
#include <span>
#include <cuda_runtime.h>
#include "parse_num.hpp"
#include "threads.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);

  if (args.size() != 3) {
    std::println(stderr, "Usage: {} <grid size> <block size>", args[0]);
    std::exit(EXIT_FAILURE);
  }

  uint32_t grid_size{};
  auto grid_size_err = parse_num(grid_size, args[1]);
  if (grid_size_err != 0) {
    std::println(stderr, "Invalid grid size: {}", std::strerror(grid_size_err));
    std::exit(EXIT_FAILURE);
  }

  uint32_t block_size{};
  auto block_size_err = parse_num(block_size, args[2]);
  if (block_size_err != 0) {
    std::println(stderr, "Invalid block size: {}", std::strerror(block_size_err));
    std::exit(EXIT_FAILURE);
  }

  auto kernel_err = threads(grid_size, block_size);
  if (kernel_err != cudaSuccess) {
    std::println(stderr, "Kernel failed: {}", cudaGetErrorString(kernel_err));
    std::exit(EXIT_FAILURE);
  }

  return EXIT_SUCCESS;
}

Writing threads.cu


In [8]:
%%bash
nvcc -std=c++23 -arch=native -O3 "threads.cu" -o threads

In [9]:
%%bash
./threads 2 3

Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from threa

d 1!
Hello from thread 2!


## 102: Extending Python with C

In [10]:
%%writefile threads_lib.cu
#include <cstdint>
#include <cuda_runtime.h>
#include "threads.cuh"

extern "C" {
  cudaError_t c_threads(uint32_t grid_size, uint32_t block_size) {
    return threads(grid_size, block_size);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing threads_lib.cu


In [11]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared threads_lib.cu -o threads_lib.so

In [12]:
%%writefile threads.py
import sys
from ctypes import (
    CDLL,
    c_char_p,
    c_int,
    c_uint32,
)
from pathlib import Path

try:
    lib = CDLL(str(Path(__file__).parent / 'threads_lib.so'))
except OSError as e:
    raise RuntimeError("Failed to load threads_lib.so") from e

lib.c_threads.argtypes = [c_uint32, c_uint32]
lib.c_threads.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def threads(grid_size: int, block_size: int) -> None:
    if (err := lib.c_threads(grid_size, block_size)) != 0:
        raise RuntimeError(f"Kernel failed: {get_error_string(err)}")

if __name__ == '__main__':
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <grid size> <block size>", file=sys.stderr)
        sys.exit(1)

    grid_size = int(sys.argv[1])
    block_size = int(sys.argv[2])
    threads(grid_size, block_size)

Writing threads.py


In [13]:
%%bash
python3 threads.py 2 3

Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from threa

d 1!
Hello from thread 2!


## 103: Hashing Strings

In [14]:
%%writefile md5.cuh
#pragma once
#include <cstdint>
#include <cuda/std/bit>
#include <cuda/std/cstring>

constexpr uint32_t md5_digest_size = 16;
constexpr uint32_t md5_block_size = 64;

__device__
inline void md5(uint8_t* digest, const uint8_t* msg, uint64_t len) {
  const uint32_t s[64] = {
    7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,  7, 12, 17, 22,
    5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,  5,  9, 14, 20,
    4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,  4, 11, 16, 23,
    6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21,  6, 10, 15, 21
  };
  const uint32_t K[64] = {
    0xd76aa478, 0xe8c7b756, 0x242070db, 0xc1bdceee,
    0xf57c0faf, 0x4787c62a, 0xa8304613, 0xfd469501,
    0x698098d8, 0x8b44f7af, 0xffff5bb1, 0x895cd7be,
    0x6b901122, 0xfd987193, 0xa679438e, 0x49b40821,
    0xf61e2562, 0xc040b340, 0x265e5a51, 0xe9b6c7aa,
    0xd62f105d, 0x02441453, 0xd8a1e681, 0xe7d3fbc8,
    0x21e1cde6, 0xc33707d6, 0xf4d50d87, 0x455a14ed,
    0xa9e3e905, 0xfcefa3f8, 0x676f02d9, 0x8d2a4c8a,
    0xfffa3942, 0x8771f681, 0x6d9d6122, 0xfde5380c,
    0xa4beea44, 0x4bdecfa9, 0xf6bb4b60, 0xbebfbc70,
    0x289b7ec6, 0xeaa127fa, 0xd4ef3085, 0x04881d05,
    0xd9d4d039, 0xe6db99e5, 0x1fa27cf8, 0xc4ac5665,
    0xf4292244, 0x432aff97, 0xab9423a7, 0xfc93a039,
    0x655b59c3, 0x8f0ccc92, 0xffeff47d, 0x85845dd1,
    0x6fa87e4f, 0xfe2ce6e0, 0xa3014314, 0x4e0811a1,
    0xf7537e82, 0xbd3af235, 0x2ad7d2bb, 0xeb86d391
  };

  uint64_t bit_len = len * 8;
  uint64_t padded_len = ((len + sizeof(bit_len)) / md5_block_size + 1) * md5_block_size;

  uint32_t a0 = 0x67452301, b0 = 0xefcdab89, c0 = 0x98badcfe, d0 = 0x10325476;
  for (uint64_t start = 0; start < padded_len; start += md5_block_size) {
    uint8_t block[md5_block_size];
    for (uint32_t i = 0; i < md5_block_size; i++) {
      uint64_t pos = start + i;
      block[i] = pos < len ? msg[pos] : pos == len ? 0x80 : 0;
    }
    if (start + md5_block_size == padded_len) {
      cuda::std::memcpy(block + md5_block_size - sizeof(bit_len), &bit_len, sizeof(bit_len));
    }

    uint32_t M[16];
    cuda::std::memcpy(M, block, sizeof(M));

    uint32_t A = a0, B = b0, C = c0, D = d0;
    for (uint32_t i = 0; i < 64; i++) {
      uint32_t F, g;
      if (i < 16) {
        F = (B & C) | (~B & D);
        g = i;
      } else if (i < 32) {
        F = (D & B) | (~D & C);
        g = (5 * i + 1) % 16;
      } else if (i < 48) {
        F = B ^ C ^ D;
        g = (3 * i + 5) % 16;
      } else {
        F = C ^ (B | ~D);
        g = (7 * i) % 16;
      }
      F = F + A + K[i] + M[g];
      A = D;
      D = C;
      C = B;
      B = B + cuda::std::rotl(F, s[i]);
    }
    a0 += A;
    b0 += B;
    c0 += C;
    d0 += D;
  }

  uint32_t out[4] = {a0, b0, c0, d0};
  cuda::std::memcpy(digest, out, sizeof(out));
}

Writing md5.cuh


In [15]:
%%writefile key_search_kernel.cuh
#pragma once
#include <cstdint>
#include <cuda/std/algorithm>
#include <cuda/std/array>
#include <cuda/std/charconv>
#include "get_thread.cuh"
#include "md5.cuh"

__global__
static void key_search_kernel(uint32_t* value, bool* found, const cuda::std::array<uint8_t, md5_digest_size> target_digest) {
  if (*found) {
    return;
  }
  
  uint32_t guess = get_thread();

  constexpr int max_digits = 10;
  char char_arr[max_digits];
  auto uint8_arr = reinterpret_cast<const uint8_t*>(char_arr);
  auto [format_end, format_err] = cuda::std::to_chars(char_arr, char_arr + max_digits, guess);

  uint8_t value_digest[md5_digest_size];
  md5(value_digest, uint8_arr, format_end - char_arr);

  if (!cuda::std::equal(value_digest, value_digest + md5_digest_size, target_digest.begin())) {
    return;
  }

  *value = guess;
  *found = true;
}

Writing key_search_kernel.cuh


In [16]:
%%writefile key_search.cuh
#pragma once
#include <cstdint>
#include <cuda/std/algorithm>
#include <cuda/std/array>
#include <cuda_runtime.h>
#include "key_search_kernel.cuh"

static inline cudaError_t key_search(uint32_t* value_host, bool* found_host, const uint8_t* target) {
  constexpr uint32_t device_id = 0;

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    return set_device_err;
  }

  cudaDeviceProp prop{};
  auto prop_err = cudaGetDeviceProperties(&prop, device_id);
  if (prop_err != cudaSuccess) {
    return prop_err;
  }

  uint32_t block_size = prop.maxThreadsPerBlock;
  uint32_t grid_size = UINT32_MAX / block_size + 1;

  cuda::std::array<uint8_t, md5_digest_size> target_host{};
  cuda::std::copy_n(target, md5_digest_size, target_host.begin());

  uint32_t* value_device{};
  bool* found_device{};

  auto value_malloc_err = cudaMallocManaged(&value_device, sizeof(uint32_t));
  if (value_malloc_err != cudaSuccess) {
    return value_malloc_err;
  }

  auto found_malloc_err = cudaMallocManaged(&found_device, sizeof(bool));
  if (found_malloc_err != cudaSuccess) {
    return found_malloc_err;
  }

  *found_device = false;

  key_search_kernel<<<grid_size, block_size>>>(value_device, found_device, target_host);
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }

  auto sync_err = cudaDeviceSynchronize();
  if (sync_err != cudaSuccess) {
    return sync_err;
  }

  *value_host = *value_device;
  *found_host = *found_device;

  cudaFree(value_device);
  cudaFree(found_device);

  return cudaSuccess;
}

Writing key_search.cuh


In [17]:
%%writefile key_search_lib.cu
#include <cstdint>
#include <cuda_runtime.h>
#include "key_search.cuh"

extern "C" {
  cudaError_t c_key_search(uint32_t* value, bool* found, const uint8_t* target) {
    return key_search(value, found, target);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing key_search_lib.cu


In [18]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared key_search_lib.cu -o key_search_lib.so

In [19]:
%%writefile key_search.py
import sys
from ctypes import (
    CDLL,
    POINTER,
    byref,
    c_bool,
    c_char_p,
    c_int,
    c_uint8,
    c_uint32,
)
from pathlib import Path

try:
    lib = CDLL(str(Path(__file__).parent / 'key_search_lib.so'))
except OSError as e:
    raise RuntimeError("Failed to load key_search_lib.so") from e

lib.c_key_search.argtypes = [POINTER(c_uint32), POINTER(c_bool), POINTER(c_uint8)]
lib.c_key_search.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def parse_hash(text: str) -> bytes:
    if len(text) != 32:
        raise ValueError(f"Invalid hash: expected 32 hex characters, got {len(text)}")
    try:
        return bytes.fromhex(text)
    except ValueError:
        raise ValueError(f"Invalid hash: {text} is not hexadecimal")

def key_search(target: bytes) -> int | None:
    buf = (c_uint8 * 16)(*target)
    value = c_uint32()
    found = c_bool()
    if (err := lib.c_key_search(byref(value), byref(found), buf)) != 0:
        raise RuntimeError(f"Hash failed: {get_error_string(err)}")
    if not found.value:
        return None
    return value.value

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <hash>", file=sys.stderr)
        sys.exit(1)

    target = parse_hash(sys.argv[1])
    value = key_search(target)
    if value is None:
        print("No match found", file=sys.stderr)
        sys.exit(1)

    print(value)

Writing key_search.py


In [20]:
%%bash
value=$SRANDOM
value_hash=$(echo -n $value | md5sum | awk '{print $1}')
guess=$(python3 key_search.py $value_hash)

echo "Expected: $value"
echo "Got:      $guess"

if [ "$guess" -eq "$value" ]; then
  echo "Correct!"
else
  echo "Incorrect!"
fi

Expected: 2273487891
Got:      2273487891


Correct!


## 104: Hashing Files

In [21]:
%%writefile hash_files_kernel.cuh
#pragma once
#include <cstdint>
#include "get_thread.cuh"
#include "md5.cuh"

__global__
static void hash_files_kernel(uint8_t* digests, const uint8_t* data, const uint64_t* offsets, uint32_t file_count) {
  uint32_t file = get_thread();
  if (file >= file_count) {
    return;
  }

  uint64_t len = offsets[file + 1] - offsets[file];
  md5(digests + file * md5_digest_size, data + offsets[file], len);
}

Writing hash_files_kernel.cuh


In [22]:
%%writefile hash_files.cuh
#pragma once
#include <cstdint>
#include <cuda_runtime.h>
#include "hash_files_kernel.cuh"
#include "md5.cuh"

static inline cudaError_t hash_files(uint8_t* digests_host, const uint8_t* data_host, const uint64_t* offsets_host, uint32_t file_count) {
  constexpr uint32_t device_id = 0;

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    return set_device_err;
  }

  cudaFuncAttributes attr{};
  auto attr_err = cudaFuncGetAttributes(&attr, hash_files_kernel);
  if (attr_err != cudaSuccess) {
    return attr_err;
  }

  uint32_t block_size = attr.maxThreadsPerBlock;
  uint32_t grid_size = file_count / block_size + 1;

  uint64_t digests_size = file_count * md5_digest_size;
  uint64_t data_size = offsets_host[file_count];
  uint64_t offsets_size = (file_count + 1) * sizeof(uint64_t);

  uint8_t* digests_device{};
  uint8_t* data_device{};
  uint64_t* offsets_device{};

  auto digests_malloc_err = cudaMalloc(&digests_device, digests_size);
  if (digests_malloc_err != cudaSuccess) {
    return digests_malloc_err;
  }

  auto data_malloc_err = cudaMalloc(&data_device, data_size);
  if (data_malloc_err != cudaSuccess) {
    return data_malloc_err;
  }

  auto offsets_malloc_err = cudaMalloc(&offsets_device, offsets_size);
  if (offsets_malloc_err != cudaSuccess) {
    return offsets_malloc_err;
  }

  auto data_copy_err = cudaMemcpy(data_device, data_host, data_size, cudaMemcpyHostToDevice);
  if (data_copy_err != cudaSuccess) {
    return data_copy_err;
  }

  auto offsets_copy_err = cudaMemcpy(offsets_device, offsets_host, offsets_size, cudaMemcpyHostToDevice);
  if (offsets_copy_err != cudaSuccess) {
    return offsets_copy_err;
  }

  hash_files_kernel<<<grid_size, block_size>>>(digests_device, data_device, offsets_device, file_count);
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }

  auto sync_err = cudaDeviceSynchronize();
  if (sync_err != cudaSuccess) {
    return sync_err;
  }

  auto digests_copy_err = cudaMemcpy(digests_host, digests_device, digests_size, cudaMemcpyDeviceToHost);
  if (digests_copy_err != cudaSuccess) {
    return digests_copy_err;
  }

  cudaFree(digests_device);
  cudaFree(data_device);
  cudaFree(offsets_device);

  return cudaSuccess;
}

Writing hash_files.cuh


In [23]:
%%writefile hash_files_lib.cu
#include <cstdint>
#include <cuda_runtime.h>
#include "hash_files.cuh"

extern "C" {
  cudaError_t c_hash_files(uint8_t* digests, const uint8_t* data, const uint64_t* offsets, uint32_t file_count) {
    return hash_files(digests, data, offsets, file_count);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}

Writing hash_files_lib.cu


In [24]:
%%bash
nvcc -std=c++23 -arch=native -O3 --compiler-options '-fPIC' -shared hash_files_lib.cu -o hash_files_lib.so

In [25]:
%%writefile hash_files_gpu.py
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from ctypes import (
    CDLL,
    POINTER,
    c_char_p,
    c_int,
    c_uint8,
    c_uint32,
    c_uint64,
)
from itertools import accumulate
from pathlib import Path

try:
    lib = CDLL(str(Path(__file__).parent / 'hash_files_lib.so'))
except OSError as e:
    raise RuntimeError("Failed to load hash_files_lib.so") from e

Digest = c_uint8 * 16

lib.c_hash_files.argtypes = [POINTER(Digest), c_char_p, POINTER(c_uint64), c_uint32]
lib.c_hash_files.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def hash_files(paths: list[str]) -> list[str]:
    contents = [Path(path).read_bytes() for path in paths]
    data = b''.join(contents)
    offsets = list(accumulate(map(len, contents), initial=0))
    offsets_arr = (c_uint64 * len(offsets))(*offsets)
    digests = (Digest * len(contents))()
    if (err := lib.c_hash_files(digests, data, offsets_arr, len(contents))) != 0:
        raise RuntimeError(f"Hash failed: {get_error_string(err)}")
    return [bytes(digest).hex() for digest in digests]

if __name__ == '__main__':
    paths = sys.stdin.read().splitlines()
    if not paths:
        print(f"Usage: find <dir> -type f | python3 {sys.argv[0]}", file=sys.stderr)
        sys.exit(1)

    paths.sort(key=os.path.getsize)
    workers = os.cpu_count()
    chunks = [paths[i::workers] for i in range(workers)]
    with ProcessPoolExecutor(workers) as pool:
        for chunk, digests in zip(chunks, pool.map(hash_files, chunks)):
            for path, digest in zip(chunk, digests):
                print(f"{digest}  {path}")

Writing hash_files_gpu.py


In [26]:
%%writefile hash_files_cpu.sh
xargs -d '\n' -P "$(nproc)" -n 64 stdbuf -oL md5sum

Writing hash_files_cpu.sh


In [27]:
%%bash
find /usr/share/man -type f -readable 2>/dev/null > paths.txt
echo "Files: $(wc -l < paths.txt)"
xargs -d '\n' cat < paths.txt > /dev/null

time bash hash_files_cpu.sh < paths.txt > cpu.txt
time python3 hash_files_gpu.py < paths.txt > gpu.txt

if diff <(sort cpu.txt) <(sort gpu.txt) > /dev/null; then
  echo "Correct!"
else
  echo "Incorrect!"
fi

Files: 6598



real	0m0.074s
user	0m0.097s
sys	0m0.288s



real	0m2.312s
user	0m0.101s
sys	0m0.050s


Correct!
